# Computer Vision Primer

This notebook is structured for industry-ready Computer Vision engineering.

Modules:
1. Image Fundamentals
2. Preprocessing Pipelines
3. Masking & Detection Utilities
4. PyTorch Production Pipeline
5. Keras Production Pipeline
6. Video Processing
7. Performance Engineering


In [ ]:
# Uncomment if needed
# !pip install opencv-python pillow matplotlib torch torchvision tensorflow

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

plt.rcParams['figure.figsize'] = (6,6)


## Module 1: Image Fundamentals

Understanding formats, color schemes, and image structure.

In [ ]:
img_path = "sample.jpg"  # Change to your image path
img = Image.open(img_path)

print("Format:", img.format)
print("Mode:", img.mode)
print("Size:", img.size)

plt.imshow(img)
plt.axis("off")
plt.show()


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img_jpg = Image.open("image.jpg")
img_png = Image.open("image.png")

plt.imshow(img_jpg)
plt.title("JPG Image")
plt.show()

plt.imshow(img_png)
plt.title("PNG Image")
plt.show()

### RGB vs BGR

- RGB (default)
- BGR (OpenCV)

In [ ]:
img_cv = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)

plt.imshow(img_rgb)
plt.title("RGB Image")
plt.axis("off")
plt.show()

print("Shape:", img_rgb.shape)


In [ ]:
from PIL import Image

img = Image.open("image.jpg")
img_resized = img.resize((224,224))
img_resized.save("resized.jpg")

In [ ]:
import cv2

img = cv2.imread("image.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
cv2.imwrite("gray.jpg", gray)

In [ ]:
# Cropping an image using OpenCV
cropped = img[100:300, 200:400]
flipped = cv2.flip(img, 1)  # Horizontal
cv2.imwrite("cropped.jpg", cropped)
cv2.imwrite("flipped.jpg", flipped)

In [ ]:
resized = cv2.resize(img, (224,224))
cv2.imwrite("resized_cv.jpg", resized)

In [ ]:
# Normalization
img = img / 255.0
print("Pixel range:", img.min(), "to", img.max())

## Module 2: Preprocessing Pipeline

In [ ]:
TARGET_SIZE = (224,224)

resized = cv2.resize(img_rgb, TARGET_SIZE)
plt.imshow(resized)
plt.title("Resized Image")
plt.axis("off")
plt.show()

In [ ]:
img_float = resized.astype(np.float32) / 255.0

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

img_normalized = (img_float - mean) / std
print("Min:", img_normalized.min(), "Max:", img_normalized.max())


In [ ]:
roi = img_rgb[100:300, 100:300]

plt.imshow(roi)
plt.title("Region of Interest")
plt.axis("off")
plt.show()


In [ ]:
flipped = cv2.flip(img_rgb, 1)

plt.imshow(flipped)
plt.title("Horizontal Flip")
plt.axis("off")
plt.show()


In [ ]:
# RGB Normalization
import torchvision.transforms as transforms

transform = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)


## Module 3: Masking & Detection Utilities

In [ ]:
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
mask = gray > 120

masked_img = img_rgb * mask[:,:,None]

plt.imshow(masked_img)
plt.title("Masked Output")
plt.axis("off")
plt.show()


In [ ]:
bbox_img = img_rgb.copy()

cv2.rectangle(bbox_img, (50,50), (200,200), (0,255,0), 2)
cv2.putText(bbox_img, "Object", (50,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (255,0,0), 2)

plt.imshow(bbox_img)
plt.axis("off")
plt.show()


In [ ]:
# Drawing bounding boxes and annotations
cv2.rectangle(img, (50,50), (200,200), (0,255,0), 2)
cv2.putText(img, "Object", (50,40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8, (255,0,0), 2)

## Module 4: PyTorch Production Pipeline

In [ ]:
import torch
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

img_tensor = train_transforms(Image.open(img_path))
print("Tensor Shape:", img_tensor.shape)


## Module 5: Keras Production Pipeline

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow_from_directory(
    "dataset/train",
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical'
)


## Module 6: Video Processing

In [ ]:
cap = cv2.VideoCapture("video.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
print("FPS:", fps)
cap.release()


In [ ]:
cap = cv2.VideoCapture("video.mp4")

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (640,480))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)

cap.release()
out.release()


In [ ]:
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    cv2.imshow("Camera", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## Capture Data From Browser

## app.py
<!-- pip install flask opencv-python numpy -->
<!-- python app.py -->



In [ ]:
from flask import Flask, request, jsonify
import base64
import numpy as np
import cv2

app = Flask(__name__)

@app.route("/upload", methods=["POST"])
def upload():

    data = request.json["image"]

    # Remove base64 header
    encoded_data = data.split(",")[1]

    # Decode base64
    decoded = base64.b64decode(encoded_data)

    # Convert to numpy
    np_arr = np.frombuffer(decoded, np.uint8)

    # Decode image
    img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

    print("Image received with shape:", img.shape)

    return jsonify({"message": "Image received!"})

if __name__ == "__main__":
    app.run(debug=True)


In [ ]:
<!DOCTYPE html>
<html>
<body>

<h2>Simple Webcam Capture</h2>

<video id="video" width="400" autoplay></video>
<br>
<button onclick="capture()">Capture & Send</button>

<script>
const video = document.getElementById("video");

// Access camera
navigator.mediaDevices.getUserMedia({ video: true })
  .then(stream => {
      video.srcObject = stream;
  });

// Capture image and send
function capture() {

    const canvas = document.createElement("canvas");
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    const ctx = canvas.getContext("2d");
    ctx.drawImage(video, 0, 0);

    const imageData = canvas.toDataURL("image/jpeg");

    fetch("http://127.0.0.1:5000/upload", {
        method: "POST",
        headers: {"Content-Type": "application/json"},
        body: JSON.stringify({ image: imageData })
    })
    .then(res => res.json())
    .then(data => alert(data.message));
}
</script>

</body>
</html>
# Browser → Capture → POST → Flask → OpenCV → Done